# Benchmark: Navegación Semántica por Comandos de Voz

Evalúa tres estrategias de recuperación semántica ante **comandos de voz** sobre un
mapa topológico de 24 nodos en 6 habitaciones.

| # | Método | Estrategia de Retrieval |
|---|--------|--------------------------|
| 1 | **SigLIP Baseline** | texto → embedding SigLIP → ANN sobre embeddings de imagen |
| 2 | **SigLIP + YOLO** | texto → embedding SigLIP + re-ranking por objetos YOLO |
| 3 | **Qwen2-VL** | texto → embedding SigLIP → ANN sobre embeddings de descripciones Qwen |

**Niveles de comando de voz:**
| Nivel | Descripción | Ejemplo |
|-------|-------------|---------|
| **L1** | Solo nombre de habitación | `"kitchen"` |
| **L2** | Habitación + 1 objeto ancla | `"kitchen with a refrigerator"` |
| **L3** | Habitación + objetos + contexto espacial | `"kitchen with a refrigerator on the right and a microwave above"` |


In [ ]:
import os, sys

# FIX para libnvrtc-builtins.so.13.0 en PyTorch + CUDA 13
_site_packages = next((p for p in sys.path if "site-packages" in p), None)
if _site_packages:
    _cu13_lib = os.path.join(_site_packages, "nvidia", "cu13", "lib")
    if os.path.exists(_cu13_lib):
        os.environ["LD_LIBRARY_PATH"] = _cu13_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Preload libnvrtc-builtins para Qwen2-VL
import ctypes
try:
    _nvrtc_path = os.path.join(_cu13_lib, "libnvrtc-builtins.so.13.0")
    if os.path.exists(_nvrtc_path):
        ctypes.CDLL(_nvrtc_path, mode=ctypes.RTLD_GLOBAL)
except Exception:
    pass

In [ ]:
# ============================================================
# Imports y verificación del entorno
# ============================================================
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import chromadb
import transformers

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"ChromaDB    : {chromadb.__version__}")
print(f"Dispositivo : {'CUDA — ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

---
## Sección 0: Dataset — Mapa Topológico

Se construye un grafo topológico con **24 nodos** en **6 habitaciones**
(`bathroom`, `bedroom`, `children_room`, `garage`, `kitchen`, `livingroom`)
a partir de las imágenes reales en `./siglip/images/`.
Por cada habitación se indexan **4 imágenes como nodos del mapa** (índices 0–3).

In [ ]:
# ============================================================
# 0.1  Directorios y parámetros globales
# ============================================================
WORK_DIR          = os.path.abspath("./benchmark_data")
IMAGE_SOURCES_DIR = os.path.abspath("./siglip/images")
CHROMA_DIR        = os.path.join(WORK_DIR, "chroma_db")
NODES_PER_ROOM    = 4

os.makedirs(CHROMA_DIR, exist_ok=True)

assert os.path.isdir(IMAGE_SOURCES_DIR), (
    f"No se encontró el directorio de imágenes: {IMAGE_SOURCES_DIR}"
)

# ============================================================
# 0.2  Habitaciones y rutas de imagen
# ============================================================
ROOMS = sorted([
    d for d in os.listdir(IMAGE_SOURCES_DIR)
    if os.path.isdir(os.path.join(IMAGE_SOURCES_DIR, d))
])
print(f"Habitaciones encontradas ({len(ROOMS)}): {ROOMS}")


def get_room_images(room_id: str) -> list:
    """Devuelve lista ordenada de rutas de imagen para una habitación."""
    _room_dir = os.path.join(IMAGE_SOURCES_DIR, room_id)
    return sorted([
        os.path.join(_room_dir, f)
        for f in os.listdir(_room_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])


# ============================================================
# 0.3  DataFrame del mapa semántico
# ============================================================
_rows   = []
_rng    = np.random.default_rng(42)

for _r_idx, _room_id in enumerate(ROOMS):
    _imgs = get_room_images(_room_id)
    for _n in range(NODES_PER_ROOM):
        _node_id = f"node_{_r_idx * NODES_PER_ROOM + _n:03d}"
        _x       = _r_idx * 3.0 + float(_rng.uniform(-0.4,  0.4))
        _y       = _n    * 1.5  + float(_rng.uniform(-0.25, 0.25))
        _rows.append({
            "node_id":    _node_id,
            "x":          round(_x, 2),
            "y":          round(_y, 2),
            "room_id":    _room_id,
            "image_path": _imgs[_n],
        })

df_map  = pd.DataFrame(_rows)
N_NODES = len(df_map)
N_ROOMS = df_map["room_id"].nunique()

print(f"\nMapa semántico: {N_NODES} nodos — {N_ROOMS} habitaciones")
display(df_map.groupby("room_id").size().rename("nodos").to_frame().T)

# ============================================================
# 0.4  Cliente ChromaDB
# ============================================================
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
print(f"\nChromaDB inicializado en: {CHROMA_DIR}")

---
## Sección 1: Carga de Modelos

Se cargan los tres modelos necesarios para construir el mapa:
- **SigLIP** (`google/siglip-base-patch16-224`) — embeddings visuales y de texto.
- **YOLOv8n** — detección de objetos para metadatos de la colección `siglip_yolo`.
- **Qwen2-VL-2B-Instruct** — descripción textual de escenas para `qwen_baseline`.

In [ ]:
# ============================================================
# 1.1  SigLIP — encoder visual y textual
# ============================================================
from transformers import AutoProcessor, AutoModel

SIGLIP_MODEL_ID = "google/siglip-base-patch16-224"
print(f"Cargando {SIGLIP_MODEL_ID} ...")
_t0 = time.perf_counter()
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID, use_fast=False)
siglip_model     = AutoModel.from_pretrained(SIGLIP_MODEL_ID)
siglip_model.eval().to(DEVICE)
print(f"  Listo en {time.perf_counter() - _t0:.1f}s  |  "
      f"Dim: {siglip_model.config.vision_config.hidden_size}  |  {DEVICE}")


@torch.no_grad()
def embed_image_siglip(image_path: str) -> list:
    """Embedding visual normalizado (L2) con SigLIP."""
    _img    = Image.open(image_path).convert("RGB")
    _inputs = siglip_processor(images=_img, return_tensors="pt")
    _inputs = {k: v.to(DEVICE) for k, v in _inputs.items()}
    _feats  = siglip_model.get_image_features(**_inputs)
    _emb    = _feats.pooler_output.squeeze(0).cpu().numpy()
    return (_emb / (np.linalg.norm(_emb) + 1e-8)).tolist()


@torch.no_grad()
def embed_text_siglip(text: str) -> list:
    """Embedding textual normalizado (L2) con SigLIP."""
    _inputs = siglip_processor(
        text=[text], return_tensors="pt", padding="max_length", truncation=True
    )
    _inputs = {k: v.to(DEVICE) for k, v in _inputs.items()}
    _feats  = siglip_model.get_text_features(**_inputs)
    _emb    = _feats.pooler_output.squeeze(0).cpu().numpy()
    return (_emb / (np.linalg.norm(_emb) + 1e-8)).tolist()


_e = embed_image_siglip(df_map["image_path"].iloc[0])
print(f"\nTest SigLIP OK — dim: {len(_e)}, ||e||: {np.linalg.norm(_e):.4f}")

In [ ]:
# ============================================================
# 1.2  YOLOv8n — detección de objetos COCO
# ============================================================
from ultralytics import YOLO

print("Cargando YOLOv8n ...")
yolo_model = YOLO("yolov8n.pt")
print(f"  YOLOv8n listo  |  {len(yolo_model.names)} clases COCO")


def detect_objects_yolo(image_path: str, conf: float = 0.20) -> list:
    """Devuelve lista de clases detectadas (sin duplicados, orden alfab.)."""
    _results = yolo_model(image_path, conf=conf, verbose=False)
    _objects = set()
    for _r in _results:
        for _cls_id in _r.boxes.cls.cpu().tolist():
            _objects.add(yolo_model.names[int(_cls_id)])
    return sorted(_objects)


print("\nMuestra de detecciones YOLO:")
for _rid in list(ROOMS)[:3]:
    _p = df_map[df_map["room_id"] == _rid]["image_path"].iloc[0]
    _o = detect_objects_yolo(_p)
    print(f"  {_rid:14s}: {_o if _o else ['(ninguno)']}")

In [ ]:
# ============================================================
# 1.3  Qwen2-VL-2B-Instruct — descripción de escenas
# ============================================================
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor as QwenProc

QWEN_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"Cargando {QWEN_MODEL_ID} ...")
_t0 = time.perf_counter()
_qwen_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
qwen_model  = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, torch_dtype=_qwen_dtype, device_map="auto",
)
qwen_model.eval()
qwen_processor = QwenProc.from_pretrained(QWEN_MODEL_ID, use_fast=False)
_n_params = sum(p.numel() for p in qwen_model.parameters()) / 1e9
print(f"  Listo en {time.perf_counter() - _t0:.1f}s  |  ~{_n_params:.1f}B  |  {_qwen_dtype}")


try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    def process_vision_info(messages):
        _imgs, _vids = [], []
        for _msg in messages:
            for _item in _msg.get("content", []):
                if _item["type"] == "image": _imgs.append(_item["image"])
                elif _item["type"] == "video": _vids.append(_item["video"])
        return _imgs or None, _vids or None


_QWEN_PROMPT = (
    "You are an AI assistant for a robot navigating indoor environments. "
    "Describe the scene briefly, listing the main objects, "
    "their spatial relationship (e.g. 'a sofa next to a table'), "
    "and the likely type of room."
)


def describe_image_qwen(image_path: str, max_new_tokens: int = 50) -> str:
    """Genera una descripción semántica de la imagen con Qwen2-VL."""
    _img = Image.open(image_path).convert("RGB")
    _messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": _img},
            {"type": "text",  "text":  _QWEN_PROMPT},
        ]
    }]
    _text       = qwen_processor.apply_chat_template(
        _messages, tokenize=False, add_generation_prompt=True
    )
    _img_inputs, _vid_inputs = process_vision_info(_messages)
    _inputs = qwen_processor(
        text=[_text], images=_img_inputs, videos=_vid_inputs,
        padding=True, return_tensors="pt",
    )
    _dev    = next(qwen_model.parameters()).device
    _inputs = {k: v.to(_dev) for k, v in _inputs.items()}
    with torch.no_grad():
        with torch.jit.fuser("fuser0"):
            _out_ids = qwen_model.generate(**_inputs, max_new_tokens=max_new_tokens)
    _generated = _out_ids[0][_inputs["input_ids"].shape[1]:]
    return qwen_processor.decode(_generated, skip_special_tokens=True).strip()


print("\nTest Qwen2-VL ...")
_t0 = time.perf_counter()
_desc_test = describe_image_qwen(df_map["image_path"].iloc[0])
print(f'  "{_desc_test}"  ({(time.perf_counter()-_t0)*1000:.0f} ms)')

---
## Sección 2: Construcción del Mapa Semántico (ChromaDB)

Se construyen las tres colecciones que sirven de base para los experimentos.
**Si una colección ya existe y está completa (`count == N_NODES`), se omite su reconstrucción.**

| Colección | Contenido | Pipeline |
|-----------|-----------|----------|
| `siglip_baseline` | Embeddings de imagen (768-dim) | imagen → SigLIP Vision |
| `siglip_yolo` | Embeddings de imagen + metadata de objetos | imagen → SigLIP Vision + YOLO |
| `qwen_baseline` | Embeddings de texto de descripciones Qwen | imagen → Qwen2-VL → SigLIP Text |

In [ ]:
# ============================================================
# 2.1  Colección: siglip_baseline
# ============================================================
_existing = {c.name for c in chroma_client.list_collections()}

if "siglip_baseline" in _existing and \
        chroma_client.get_collection("siglip_baseline").count() == N_NODES:
    collection_siglip = chroma_client.get_collection("siglip_baseline")
    print(f"siglip_baseline: ya construida ({N_NODES} nodos) — omitiendo.")
else:
    if "siglip_baseline" in _existing:
        chroma_client.delete_collection("siglip_baseline")
    collection_siglip = chroma_client.create_collection(
        name="siglip_baseline", metadata={"hnsw:space": "cosine"}
    )
    print("Construyendo siglip_baseline ...")
    _t0 = time.perf_counter()
    _embs, _ids, _metas = [], [], []
    for _, _row in df_map.iterrows():
        _embs.append(embed_image_siglip(_row["image_path"]))
        _ids.append(_row["node_id"])
        _metas.append({"room_id": _row["room_id"], "x": _row["x"], "y": _row["y"]})
    collection_siglip.upsert(ids=_ids, embeddings=_embs, metadatas=_metas)
    print(f"  Listo en {time.perf_counter()-_t0:.2f}s  ({N_NODES} nodos)")

In [ ]:
# ============================================================
# 2.2  Colección: siglip_yolo
# ============================================================
_existing = {c.name for c in chroma_client.list_collections()}

if "siglip_yolo" in _existing and \
        chroma_client.get_collection("siglip_yolo").count() == N_NODES:
    collection_siglip_yolo = chroma_client.get_collection("siglip_yolo")
    print(f"siglip_yolo: ya construida ({N_NODES} nodos) — omitiendo.")
else:
    if "siglip_yolo" in _existing:
        chroma_client.delete_collection("siglip_yolo")
    collection_siglip_yolo = chroma_client.create_collection(
        name="siglip_yolo", metadata={"hnsw:space": "cosine"}
    )
    print("Construyendo siglip_yolo (SigLIP + YOLO) ...")
    _t0 = time.perf_counter()
    _embs, _ids, _metas = [], [], []
    for _, _row in df_map.iterrows():
        _objs = detect_objects_yolo(_row["image_path"])
        _embs.append(embed_image_siglip(_row["image_path"]))
        _ids.append(_row["node_id"])
        _metas.append({
            "room_id": _row["room_id"],
            "x":       _row["x"],
            "y":       _row["y"],
            "objects": ",".join(_objs) if _objs else "__none__",
        })
    collection_siglip_yolo.upsert(ids=_ids, embeddings=_embs, metadatas=_metas)
    print(f"  Listo en {time.perf_counter()-_t0:.2f}s  ({N_NODES} nodos)")

In [ ]:
# ============================================================
# 2.3  Colección: qwen_baseline
#      Pipeline: imagen → Qwen2-VL (desc.) → SigLIP Text (emb.)
# ============================================================
_existing = {c.name for c in chroma_client.list_collections()}

if "qwen_baseline" in _existing and \
        chroma_client.get_collection("qwen_baseline").count() == N_NODES:
    collection_qwen = chroma_client.get_collection("qwen_baseline")
    print(f"qwen_baseline: ya construida ({N_NODES} nodos) — omitiendo.")
else:
    if "qwen_baseline" in _existing:
        chroma_client.delete_collection("qwen_baseline")
    collection_qwen = chroma_client.create_collection(
        name="qwen_baseline", metadata={"hnsw:space": "cosine"}
    )
    print("Construyendo qwen_baseline (puede tardar varios minutos) ...")
    _t0 = time.perf_counter()
    _embs, _ids, _metas = [], [], []
    for _i, (_, _row) in enumerate(df_map.iterrows()):
        _desc = describe_image_qwen(_row["image_path"])
        _embs.append(embed_text_siglip(_desc))
        _ids.append(_row["node_id"])
        _metas.append({
            "room_id":     _row["room_id"],
            "x":           _row["x"],
            "y":           _row["y"],
            "description": _desc[:250],
        })
        if (_i + 1) % 4 == 0:
            print(f"  [{_i+1:02d}/{N_NODES}] {_row['room_id']:14s}: \"{_desc[:65]}...\"")
    collection_qwen.upsert(ids=_ids, embeddings=_embs, metadatas=_metas)
    print(f"  Listo en {time.perf_counter()-_t0:.1f}s  ({N_NODES} nodos)")

print("\nColecciones disponibles:")
for _c in [collection_siglip, collection_siglip_yolo, collection_qwen]:
    print(f"  {_c.name:25s}: {_c.count()} nodos")

---
## Sección 3: Dataset de Comandos de Voz

Comandos en **tres niveles de detalle** para las 6 habitaciones del mapa.
Los objetos en L2/L3 están alineados con las detecciones reales de YOLO
(confirmadas en la construcción del mapa).

In [ ]:
# ============================================================
# 3.1  Dataset de comandos de voz con Ground Truth
# ============================================================
# gt_node: primer nodo de cada habitación (node_000, node_004, ...)
# gt_room: nombre de la habitación  (métrica principal de navegación)

VOICE_QUERIES = {
    "bathroom": {
        "gt_node": "node_000",
        "gt_room": "bathroom",
        "L1": "bathroom",
        "L2": "bathroom with a sink and a toilet",
        "L3": "small bathroom with a white sink mounted on the left wall and a toilet next to it",
    },
    "bedroom": {
        "gt_node": "node_004",
        "gt_room": "bedroom",
        "L1": "bedroom",
        "L2": "bedroom with a bed",
        "L3": "cozy bedroom with a large bed in the center and a teddy bear resting on the pillow",
    },
    "children_room": {
        "gt_node": "node_008",
        "gt_room": "children_room",
        "L1": "children room",
        "L2": "children room with chairs and a dining table",
        "L3": "bright children room with colorful chairs around a dining table and a cup placed on the surface",
    },
    "garage": {
        "gt_node": "node_012",
        "gt_room": "garage",
        "L1": "garage",
        "L2": "garage with a car parked inside",
        "L3": "large garage with a car parked on the right side and shelves with tools on the back wall",
    },
    "kitchen": {
        "gt_node": "node_016",
        "gt_room": "kitchen",
        "L1": "kitchen",
        "L2": "kitchen with a refrigerator",
        "L3": "kitchen with a stainless steel refrigerator on the right and a microwave mounted above the counter",
    },
    "livingroom": {
        "gt_node": "node_020",
        "gt_room": "livingroom",
        "L1": "living room",
        "L2": "living room with a couch",
        "L3": "comfortable living room with a large couch facing a TV on the wall and a coffee table in front",
    },
}

LEVELS  = ["L1", "L2", "L3"]
N_VOICE = len(VOICE_QUERIES)
print(f"Habitaciones: {N_VOICE}   |   Niveles: {LEVELS}")

In [ ]:
# ============================================================
# 3.2  Visualización del dataset
# ============================================================
_rows_display = []
for _room, _data in VOICE_QUERIES.items():
    for _lvl in LEVELS:
        _rows_display.append({
            "Habitación": _room,
            "GT Node":    _data["gt_node"],
            "Nivel":      _lvl,
            "Comando":    _data[_lvl],
        })
df_voice = pd.DataFrame(_rows_display)

pd.set_option("display.max_colwidth", 95)
display(df_voice.pivot(index="Habitación", columns="Nivel", values="Comando"))

---
## Sección 4: Experimento 1 — SigLIP Baseline (Text-to-Image)

El comando de voz se convierte en embedding con el **encoder de texto de SigLIP**
y se consulta contra `siglip_baseline` (embeddings visuales).
Evalúa el alineamiento nativo texto-imagen de SigLIP sin ningún post-procesamiento.

```
voz ──[SigLIP Text]──> emb_texto ──[ANN coseno]──> siglip_baseline (imágenes)
```

In [ ]:
# ============================================================
# 4.1  Experimento SigLIP Baseline — Text → Image
# ============================================================
print("Ejecutando SigLIP Baseline (Text → Image) ...")
print("-" * 65)

results_siglip_baseline = []

for _lvl in LEVELS:
    _ret_times = []
    _top1 = _top3 = _room = 0

    for _room_id, _qdata in VOICE_QUERIES.items():
        _t0    = time.perf_counter()
        _q_emb = embed_text_siglip(_qdata[_lvl])
        _res   = collection_siglip.query(query_embeddings=[_q_emb], n_results=3)
        _ret_times.append((time.perf_counter() - _t0) * 1000)

        _ret_ids   = _res["ids"][0]
        _ret_rooms = [_m["room_id"] for _m in _res["metadatas"][0]]

        if _ret_ids[0] == _qdata["gt_node"]:   _top1 += 1
        if _qdata["gt_node"] in _ret_ids:       _top3 += 1
        if _qdata["gt_room"] in _ret_rooms:     _room += 1

    results_siglip_baseline.append({
        "Método":                  "SigLIP Baseline",
        "Nivel":                   _lvl,
        "T. Retrieval Medio (ms)": round(float(np.mean(_ret_times)), 2),
        "T. Retrieval Mín (ms)":   round(float(np.min(_ret_times)),  2),
        "T. Retrieval Máx (ms)":   round(float(np.max(_ret_times)),  2),
        "Top-1 Acc":               round(_top1 / N_VOICE, 3),
        "Top-3 Acc":               round(_top3 / N_VOICE, 3),
        "Room Acc":                round(_room  / N_VOICE, 3),
    })
    print(f"  [{_lvl}]  T.Ret={np.mean(_ret_times):.1f}ms  "
          f"Top-1={_top1/N_VOICE:.2f}  Top-3={_top3/N_VOICE:.2f}  RoomAcc={_room/N_VOICE:.2f}")

print("\nSigLIP Baseline completado.")

---
## Sección 5: Experimento 2 — SigLIP + YOLO (Búsqueda Híbrida)

Recuperación en dos pasos:
1. **ANN visual**: comando → embedding SigLIP → top-K candidatos en `siglip_yolo`.
2. **Re-ranking**: se extraen clases de objetos COCO del comando y se bonifica a los
   nodos cuyo metadato YOLO coincida: `score = (1 − dist) + 0.08 × solapamiento`.

```
voz ──[keyword extract]──> ["refrigerator", "microwave"]   ─┐
    ──[SigLIP Text]──> emb_texto ──[ANN]──> top-K candidatos ┤
                                                              └──> re-ranking ──> top-3
```

In [ ]:
# ============================================================
# 5.1  Extracción de palabras clave COCO desde texto
# ============================================================
_COCO_CLASSES = {
    "person", "bicycle", "car", "motorcycle", "bus", "truck",
    "bench", "backpack", "umbrella", "suitcase",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl",
    "banana", "apple", "sandwich", "orange", "cake",
    "chair", "couch", "potted plant", "bed", "dining table",
    "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator",
    "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush", "cat", "dog",
}

# Sinónimos → clase COCO canónica
_SYNONYMS = {
    "sofa":         "couch",
    "television":   "tv",
    "table":        "dining table",
    "fridge":       "refrigerator",
    "coffee table": "dining table",
    "mug":          "cup",
    "phone":        "cell phone",
}


def extract_yolo_keywords(text: str) -> list:
    """Extrae clases COCO mencionadas en el texto (con resolución de sinónimos)."""
    _t = text.lower()
    for _syn, _canon in _SYNONYMS.items():
        _t = _t.replace(_syn, _canon)
    _found = set()
    for _cls in sorted(_COCO_CLASSES, key=len, reverse=True):  # multi-word primero
        if _cls in _t:
            _found.add(_cls)
            _t = _t.replace(_cls, " ")
    return sorted(_found)


# Validación
print("Keywords extraídas por comando:")
for _room, _qdata in VOICE_QUERIES.items():
    _kws = {_lvl: extract_yolo_keywords(_qdata[_lvl]) for _lvl in LEVELS}
    _any = any(_kws[l] for l in LEVELS)
    if _any:
        print(f"  {_room:14s}  L1={_kws['L1']}  L2={_kws['L2']}  L3={_kws['L3']}")

In [ ]:
# ============================================================
# 5.2  Experimento SigLIP + YOLO — Búsqueda Híbrida
# ============================================================
N_CAND     = min(10, collection_siglip_yolo.count())
YOLO_BONUS = 0.08

print(f"Ejecutando SigLIP + YOLO (N_CAND={N_CAND}, bonus={YOLO_BONUS}) ...")
print("-" * 65)

results_siglip_yolo = []

for _lvl in LEVELS:
    _ret_times = []
    _top1 = _top3 = _room = 0

    for _room_id, _qdata in VOICE_QUERIES.items():
        _kws = set(extract_yolo_keywords(_qdata[_lvl]))

        _t0    = time.perf_counter()
        _q_emb = embed_text_siglip(_qdata[_lvl])

        # Paso 1: ANN broad search
        _res_broad = collection_siglip_yolo.query(
            query_embeddings=[_q_emb], n_results=N_CAND
        )

        # Paso 2: re-ranking por solapamiento de objetos YOLO
        _ranked = []
        for _nid, _meta, _dist in zip(
            _res_broad["ids"][0],
            _res_broad["metadatas"][0],
            _res_broad["distances"][0],
        ):
            _node_objs = set(_meta.get("objects", "__none__").split(","))
            _overlap   = len(_kws & _node_objs) if _kws else 0
            _score     = (1.0 - _dist) + YOLO_BONUS * _overlap
            _ranked.append({"id": _nid, "room_id": _meta["room_id"], "score": _score})

        _ranked.sort(key=lambda x: x["score"], reverse=True)
        _ret_times.append((time.perf_counter() - _t0) * 1000)

        _top3_ids   = [_r["id"]      for _r in _ranked[:3]]
        _top3_rooms = [_r["room_id"] for _r in _ranked[:3]]

        if _top3_ids[0] == _qdata["gt_node"]:  _top1 += 1
        if _qdata["gt_node"] in _top3_ids:      _top3 += 1
        if _qdata["gt_room"] in _top3_rooms:    _room += 1

    results_siglip_yolo.append({
        "Método":                  "SigLIP + YOLO",
        "Nivel":                   _lvl,
        "T. Retrieval Medio (ms)": round(float(np.mean(_ret_times)), 2),
        "T. Retrieval Mín (ms)":   round(float(np.min(_ret_times)),  2),
        "T. Retrieval Máx (ms)":   round(float(np.max(_ret_times)),  2),
        "Top-1 Acc":               round(_top1 / N_VOICE, 3),
        "Top-3 Acc":               round(_top3 / N_VOICE, 3),
        "Room Acc":                round(_room  / N_VOICE, 3),
    })
    print(f"  [{_lvl}]  T.Ret={np.mean(_ret_times):.1f}ms  "
          f"Top-1={_top1/N_VOICE:.2f}  Top-3={_top3/N_VOICE:.2f}  RoomAcc={_room/N_VOICE:.2f}")

print("\nSigLIP + YOLO completado.")

---
## Sección 6: Experimento 3 — Qwen2-VL (Text-to-Text)

El comando de voz se vectoriza con el **encoder de texto de SigLIP** y se consulta
contra `qwen_baseline`, que almacena embeddings de las **descripciones textuales**
generadas por Qwen2-VL sobre las imágenes de los nodos.

Esto evalúa la **similitud semántica texto-a-texto**: ¿coincide el lenguaje natural
del comando con el lenguaje descriptivo detallado de Qwen?
No se procesa ninguna imagen en tiempo de consulta → latencia mínima.

```
voz ──[SigLIP Text]──> emb_texto ──[ANN coseno]──> qwen_baseline (descripciones Qwen)
```

In [ ]:
# ============================================================
# 6.1  Experimento Qwen2-VL — Text → Text
# ============================================================
print("Ejecutando Qwen2-VL Baseline (Text → Text) ...")
print("-" * 65)

results_qwen = []

for _lvl in LEVELS:
    _ret_times = []
    _top1 = _top3 = _room = 0

    for _room_id, _qdata in VOICE_QUERIES.items():
        _t0    = time.perf_counter()
        _q_emb = embed_text_siglip(_qdata[_lvl])
        _res   = collection_qwen.query(query_embeddings=[_q_emb], n_results=3)
        _ret_times.append((time.perf_counter() - _t0) * 1000)

        _ret_ids   = _res["ids"][0]
        _ret_rooms = [_m["room_id"] for _m in _res["metadatas"][0]]

        if _ret_ids[0] == _qdata["gt_node"]:   _top1 += 1
        if _qdata["gt_node"] in _ret_ids:       _top3 += 1
        if _qdata["gt_room"] in _ret_rooms:     _room += 1

        # Diagnóstico: descripción Qwen del nodo top-1
        if _lvl == "L3":
            _desc_top1 = _res["metadatas"][0][0].get("description", "")
            _match     = "✓" if _qdata["gt_room"] in _ret_rooms else "✗"
            print(f"  {_match} {_room_id:14s}: \"{_desc_top1[:70]}...\"")

    results_qwen.append({
        "Método":                  "Qwen2-VL",
        "Nivel":                   _lvl,
        "T. Retrieval Medio (ms)": round(float(np.mean(_ret_times)), 2),
        "T. Retrieval Mín (ms)":   round(float(np.min(_ret_times)),  2),
        "T. Retrieval Máx (ms)":   round(float(np.max(_ret_times)),  2),
        "Top-1 Acc":               round(_top1 / N_VOICE, 3),
        "Top-3 Acc":               round(_top3 / N_VOICE, 3),
        "Room Acc":                round(_room  / N_VOICE, 3),
    })
    print(f"  [{_lvl}]  T.Ret={np.mean(_ret_times):.1f}ms  "
          f"Top-1={_top1/N_VOICE:.2f}  Top-3={_top3/N_VOICE:.2f}  RoomAcc={_room/N_VOICE:.2f}")

print("\nQwen2-VL completado.")

---
## Sección 7: Resultados y Visualización

Consolidación de todos los resultados y dos gráficos:

1. **Line Chart** — Room Accuracy vs Nivel de Detalle (L1 → L2 → L3).
2. **Bar Chart** — Latencia media de retrieval por método y nivel.

In [ ]:
# ============================================================
# 7.1  DataFrame comparativo
# ============================================================
df_results = pd.DataFrame(
    results_siglip_baseline + results_siglip_yolo + results_qwen
)

DISPLAY_COLS = [
    "Método", "Nivel",
    "T. Retrieval Medio (ms)", "T. Retrieval Mín (ms)", "T. Retrieval Máx (ms)",
    "Top-1 Acc", "Top-3 Acc", "Room Acc",
]
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 130)

print("=" * 125)
print("  VOICE NAVIGATION BENCHMARK — Navegación Semántica por Comandos de Voz")
print("=" * 125)
print(df_results[DISPLAY_COLS].to_string(index=False))
print("=" * 125)
print()
print("Leyenda:")
print("  L1 = solo nombre de habitación")
print("  L2 = habitación + 1 objeto ancla")
print("  L3 = habitación + múltiples objetos + contexto espacial")
print("  Room Acc = la habitación GT aparece en el top-3 (métrica clave para navegación)")

display(df_results[DISPLAY_COLS].set_index(["Método", "Nivel"]))

In [ ]:
# ============================================================
# 7.2  Line Chart — Room Accuracy vs Nivel de Detalle
# ============================================================
PALETTE = {
    "SigLIP Baseline": "#4C72B0",
    "SigLIP + YOLO":   "#DD8452",
    "Qwen2-VL":        "#55A868",
}
MARKERS = {
    "SigLIP Baseline": "o",
    "SigLIP + YOLO":   "s",
    "Qwen2-VL":        "^",
}

fig, ax = plt.subplots(figsize=(9, 5.5))

for _method, _color in PALETTE.items():
    _df_m = df_results[df_results["Método"] == _method].sort_values("Nivel")
    _y    = _df_m["Room Acc"].tolist()
    ax.plot([0, 1, 2], _y,
            color=_color, marker=MARKERS[_method],
            linewidth=2.5, markersize=9, label=_method)
    for _xi, _yi in enumerate(_y):
        ax.annotate(f"{_yi:.2f}", xy=(_xi, _yi),
                    xytext=(0, 10), textcoords="offset points",
                    ha="center", fontsize=9, color=_color, fontweight="bold")

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(
    ["L1\n(solo nombre)", "L2\n(+ objeto ancla)", "L3\n(+ contexto espacial)"],
    fontsize=10,
)
ax.set_yticks(np.arange(0, 1.21, 0.2))
ax.set_ylim(-0.05, 1.25)
ax.set_xlabel("Nivel de Detalle del Comando de Voz", fontsize=11)
ax.set_ylabel("Room Accuracy (top-3)", fontsize=11)
ax.set_title(
    "Voice Navigation: Room Accuracy por Nivel de Detalle\n"
    "¿Qué tan bien localiza el robot la habitación correcta según el comando de voz?",
    fontsize=12, fontweight="bold",
)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="y", alpha=0.35, linestyle="--")
ax.set_axisbelow(True)

plt.tight_layout()
_out = os.path.join(WORK_DIR, "voice_nav_room_acc.png")
plt.savefig(_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {_out}")

In [ ]:
# ============================================================
# 7.3  Bar Chart — Latencia de Retrieval por método y nivel
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5.5))

_x       = np.arange(len(LEVELS))
_methods = list(PALETTE.keys())
_n       = len(_methods)
_w       = 0.22
_offsets = np.linspace(-(_n - 1) * _w / 2, (_n - 1) * _w / 2, _n)

for _i, (_method, _color) in enumerate(PALETTE.items()):
    _df_m  = df_results[df_results["Método"] == _method].sort_values("Nivel")
    _lats  = _df_m["T. Retrieval Medio (ms)"].tolist()
    _bars  = ax.bar(
        _x + _offsets[_i], _lats, _w,
        label=_method, color=_color, alpha=0.88, edgecolor="white",
    )
    ax.bar_label(_bars, labels=[f"{_v:.1f}" for _v in _lats],
                 padding=3, fontsize=8, color=_color)

ax.set_xticks(_x)
ax.set_xticklabels(
    ["L1\n(solo nombre)", "L2\n(+ objeto ancla)", "L3\n(+ contexto espacial)"],
    fontsize=10,
)
ax.set_ylabel("T. Retrieval Medio (ms)", fontsize=11)
ax.set_title(
    "Voice Navigation: Latencia de Retrieval por Método y Nivel\n"
    "(texto → embedding SigLIP → ChromaDB ANN)",
    fontsize=12, fontweight="bold",
)
ax.legend(loc="upper right", fontsize=10)
ax.grid(axis="y", alpha=0.35, linestyle="--")
ax.set_axisbelow(True)

plt.tight_layout()
_out = os.path.join(WORK_DIR, "voice_nav_latency.png")
plt.savefig(_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {_out}")

In [ ]:
# ============================================================
# 7.4  Resumen ejecutivo
# ============================================================
_summary = (
    df_results
    .groupby("Método")
    .agg(
        Room_Acc_L1=("Room Acc", lambda x: x.iloc[0]),
        Room_Acc_L2=("Room Acc", lambda x: x.iloc[1]),
        Room_Acc_L3=("Room Acc", lambda x: x.iloc[2]),
        Latencia_Media=("T. Retrieval Medio (ms)", "mean"),
    )
    .round(3)
)
_summary.columns = ["Room Acc L1", "Room Acc L2", "Room Acc L3", "Latencia Media (ms)"]

print("\n" + "=" * 72)
print("  RESUMEN EJECUTIVO — VOICE NAVIGATION BENCHMARK")
print("=" * 72)
display(_summary)

_best_l3  = _summary["Room Acc L3"].idxmax()
_best_lat = _summary["Latencia Media (ms)"].idxmin()
_best_avg = (_summary[["Room Acc L1", "Room Acc L2", "Room Acc L3"]]
             .mean(axis=1).idxmax())
print(f"\n  Mejor Room Acc en L3:          {_best_l3}")
print(f"  Mejor Room Acc media (L1–L3):  {_best_avg}")
print(f"  Menor latencia media:          {_best_lat}")
print("=" * 72)
print("\nBenchmark completado.")